# 🔧 Pipeline de Preprocesamiento — Car Price Dataset

Construimos la 'línea de ensamblaje' que limpia automáticamente los datos
para el modelo de predicción de MSRP.

**¿Por qué usar un Pipeline de Scikit-Learn?**
* Aplica todos los pasos en orden con un solo comando
* Garantiza reproducibilidad en producción
* Previene **Data Leakage**: `fit()` aprende solo de datos de entrenamiento

In [1]:
# ==========================================
# IMPORTACIONES
# ==========================================

import sys
sys.path.insert(0, '..')   # Permite importar desde la carpeta src/

import pandas as pd
import numpy as np
import os
os.makedirs('../outputs', exist_ok=True)  # Carpeta donde guardaremos los resultados

# BaseEstimator, TransformerMixin: clases base para crear transformadores propios
from sklearn.base import BaseEstimator, TransformerMixin

# Pipeline: encadena transformadores en secuencia
from sklearn.pipeline import Pipeline

# ColumnTransformer: aplica rutas distintas a numéricas y categóricas
from sklearn.compose import ColumnTransformer

# SimpleImputer: rellena NaN con mediana, moda, etc.
from sklearn.impute import SimpleImputer

# StandardScaler: estandariza a media=0, std=1
# OneHotEncoder: convierte texto a columnas de 0 y 1
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Muestra el diagrama visual del pipeline en el notebook
from sklearn import set_config
set_config(display='diagram')

print('✅ Importaciones exitosas')

✅ Importaciones exitosas


## 1. Verificación de Integridad (SHA-256)

Antes de procesar cualquier dato verificamos que el archivo no fue modificado.
El SHA-256 genera una 'huella digital' única — si alguien edita el CSV, la huella cambia.

In [2]:
import os
os.chdir('..')  # Cambia al directorio raíz del proyecto

# Importamos la función de auditoría de src/audit.py
from src.audit import audit_data

# audit_data() busca el CSV en data/raw/, calcula su hash SHA-256
# y compara con el hash guardado en metadata.json
# Primera vez: crea el metadata.json. Siguientes: verifica que no cambió
audit_data()

🔍 Auditing file: data.csv
✅ SUCCESS: Data integrity verified. File has not been altered.


True

In [3]:
# Carga y optimización de memoria
from src.optimization import optimize_memory, process_in_chunks

# Demostración de chunk processing (escabilidad para archivos grandes)
# En producción con millones de filas, no se puede cargar todo en RAM
process_in_chunks('data/raw/data.csv', chunk_size=5000)

# Cargar el dataset completo
df_raw = pd.read_csv('data/raw/data.csv')
print(f'\nDimensiones originales: {df_raw.shape}')

# optimize_memory reduce el tipo de dato de cada columna al mínimo necesario
# Year (1990-2017) pasa de int64 (8 bytes) a int16 (2 bytes) → ahorra 75% por columna
df_opt = optimize_memory(df_raw)


📦 Processing 'data/raw/data.csv' in chunks of 5000 rows...
✅ Total rows processed: 11914

Dimensiones originales: (11914, 16)
💾 Original memory usage: 6.08 MB
🚀 Optimized memory usage: 5.63 MB
📉 Total savings: 7.5%


## 2. Transformadores Personalizados (Robots de Limpieza)

Creamos clases heredando de `BaseEstimator` y `TransformerMixin`.
Esto nos obliga a implementar:
* `fit()`: **aprende** parámetros del dato de entrenamiento
* `transform()`: **aplica** lo aprendido a cualquier conjunto de datos

La separación `fit` / `transform` es lo que **previene el Data Leakage**.

In [4]:
# 🤖 ROBOT 1: El Traductor
# Problema: Transmission Type tiene 19 registros con el string 'UNKNOWN'
# Pandas no los detecta como nulos → debemos convertirlos a NaN real

class UnknownToNaNTransformer(BaseEstimator, TransformerMixin):
    """Convierte el string 'UNKNOWN' en np.nan para su posterior imputación."""

    def fit(self, X, y=None):
        return self  # No necesita aprender nada, siempre hace lo mismo

    def transform(self, X):
        X_copy = X.copy()  # SIEMPRE trabajar en copia, no modificar el original
        X_copy = X_copy.replace('UNKNOWN', np.nan)
        return X_copy

# Prueba rápida para demostrar que funciona
prueba = pd.DataFrame({'Transmission Type': ['MANUAL', 'UNKNOWN', 'AUTOMATIC']})
print('ANTES:')
print(prueba)
print('\nDESPUÉS:')
print(UnknownToNaNTransformer().transform(prueba))

ANTES:
  Transmission Type
0            MANUAL
1           UNKNOWN
2         AUTOMATIC

DESPUÉS:
  Transmission Type
0            MANUAL
1               NaN
2         AUTOMATIC


In [5]:
# 👮 ROBOT 2: El Guardia de Seguridad
# Elimina columnas que causarían problemas al modelo
# 'Model': 915 valores únicos → OneHotEncoding crearía 915 columnas nuevas (overfitting)

class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas de alta cardinalidad o con data leakage."""

    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop  # Lista de columnas a eliminar

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Solo elimina si la columna existe (evita errores con datos nuevos)
        cols = [c for c in self.columns_to_drop if c in X_copy.columns]
        return X_copy.drop(columns=cols)

In [6]:
# ✂️ ROBOT 3: La Máquina Recortadora (Winsorización)
# MSRP tiene outliers extremos (Bugatti: $2.065.902)
# En vez de eliminar esas filas, recortamos el valor al límite IQR
# El auto sigue en el dataset — solo su precio queda limitado a $74.078

class OutlierCapper(BaseEstimator, TransformerMixin):
    """Limita outliers al rango IQR sin eliminar filas."""

    def __init__(self, factor=1.5):
        self.factor = factor
        self.lower_bounds_ = {}  # Límites aprendidos en fit()
        self.upper_bounds_ = {}

    def fit(self, X, y=None):
        # MEMORIZA los límites usando SOLO datos de entrenamiento
        X_num = X.select_dtypes(include=['number'])
        for col in X_num.columns:
            Q1 = X_num[col].quantile(0.25)
            Q3 = X_num[col].quantile(0.75)
            IQR = Q3 - Q1
            self.lower_bounds_[col] = Q1 - self.factor * IQR
            self.upper_bounds_[col] = Q3 + self.factor * IQR
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in X_copy.select_dtypes(include=['number']).columns:
            if col in self.lower_bounds_:
                # np.clip: si valor < lower → sube a lower; si > upper → baja a upper
                X_copy[col] = np.clip(X_copy[col],
                                      self.lower_bounds_[col],
                                      self.upper_bounds_[col])
        return X_copy

    def get_feature_names_out(self, input_features=None):
        return input_features

# Prueba rápida
capper = OutlierCapper(factor=1.5)
df_test = capper.fit_transform(df_opt)
print(f'💰 MSRP máximo ANTES : ${df_raw["MSRP"].max():>10,.0f}  (Bugatti Veyron)')
print(f'✂️  MSRP máximo DESPUÉS: ${df_test["MSRP"].max():>10,.0f}  (límite IQR)')

💰 MSRP máximo ANTES : $ 2,065,902  (Bugatti Veyron)
✂️  MSRP máximo DESPUÉS: $    74,078  (límite IQR)


In [7]:
# 🗑️ ROBOT 4: El Limpiador de Columnas Vacías
# Market Category tiene 31.4% de nulos Y valores multi-etiqueta
# ('Luxury,Performance,High-Performance') → imposible de encodificar bien

class DropHighMissingTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas que superen un umbral de nulos."""

    def __init__(self, threshold=0.25):
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        # isnull().mean() da el porcentaje de nulos por columna (0.0 a 1.0)
        pct_nulos = X.isnull().mean()
        self.cols_to_drop_ = pct_nulos[pct_nulos > self.threshold].index.tolist()
        print(f'🗑️  Columnas eliminadas (>{self.threshold*100:.0f}% nulos): {self.cols_to_drop_}')
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)


# 🧠 ROBOT 5: El Imputador Inteligente
# Rellena nulos con mediana (números) o moda (texto), según el % de nulos
# Mediana es mejor que media cuando hay outliers (ej. Engine HP: Bugatti 1001 HP)

class SmartImputerTransformer(BaseEstimator, TransformerMixin):
    """
    Imputa según el % de nulos:
    - < 10%: mediana (numérico) o moda (categórico)
    - > 10%: pendiente (KNNImputer en el futuro)
    """

    def __init__(self, low_threshold=0.10):
        self.low_threshold = low_threshold
        self.cols_simples_   = []
        self.cols_complejas_ = []

    def fit(self, X, y=None):
        pct_nulos = X.isnull().mean()
        for col in X.columns:
            pct = pct_nulos[col]
            if 0 < pct <= self.low_threshold:
                self.cols_simples_.append(col)
            elif pct > self.low_threshold:
                self.cols_complejas_.append(col)
        print(f'🧠 SmartImputer - Simple (<10%): {self.cols_simples_}')
        print(f'🚧 SmartImputer - Complejo (>10%): {self.cols_complejas_} (PENDIENTE - futuro KNNImputer)')
        return self

    def transform(self, X):
        X_copy = X.copy()
        for col in self.cols_simples_:
            if pd.api.types.is_numeric_dtype(X_copy[col]):
                # Mediana: más robusta que la media cuando hay outliers
                X_copy[col] = X_copy[col].fillna(X_copy[col].median())
            else:
                # Moda: el valor más frecuente para variables de texto
                X_copy[col] = X_copy[col].fillna(X_copy[col].mode()[0])
        return X_copy

    def get_feature_names_out(self, input_features=None):
        return input_features

print('✅ Los 5 robots de limpieza han sido programados')

✅ Los 5 robots de limpieza han sido programados


## 3. Ensamblaje del Pipeline

Conectamos los 5 robots en secuencia — como una línea de montaje de fábrica.

In [8]:
# ==========================================
# DEFINICIÓN DE VARIABLES POR RUTA
# ==========================================

# Numéricas: van a OutlierCapper → SimpleImputer → StandardScaler
variables_numericas = ['Year', 'Engine HP', 'Engine Cylinders',
                       'Number of Doors', 'highway MPG', 'city mpg', 'Popularity']

# Categóricas: van a SimpleImputer → OneHotEncoder
# Excluimos 'Model' (alta cardinalidad) y 'Market Category' (31% nulos)
variables_categoricas = ['Make', 'Engine Fuel Type', 'Transmission Type',
                         'Driven_Wheels', 'Vehicle Size', 'Vehicle Style']

print(f'Variables numéricas  ({len(variables_numericas)}): {variables_numericas}')
print(f'Variables categóricas ({len(variables_categoricas)}): {variables_categoricas}')

Variables numéricas  (7): ['Year', 'Engine HP', 'Engine Cylinders', 'Number of Doors', 'highway MPG', 'city mpg', 'Popularity']
Variables categóricas (6): ['Make', 'Engine Fuel Type', 'Transmission Type', 'Driven_Wheels', 'Vehicle Size', 'Vehicle Style']


In [9]:
# ==========================================
# RUTAS INTERNAS (Sub-pipelines)
# ==========================================

# Ruta Numérica: capping → rellena nulos → normaliza escala
numeric_pipeline = Pipeline([
    ('capper',  OutlierCapper()),                    # Recorta extremos
    ('imputer', SimpleImputer(strategy='median')),   # Rellena NaN con mediana
    ('scaler',  StandardScaler())                    # Estandariza (media=0, std=1)
])

# Ruta Categórica: rellena nulos → codifica como 0 y 1
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),   # Moda
    ('onehot',  OneHotEncoder(handle_unknown='ignore',      # Ignora categorías nuevas
                              sparse_output=False))         # Devuelve matriz densa
])

# ColumnTransformer: enruta cada grupo a su pipeline
# remainder='drop' descarta todo lo que no está en las listas (ej. Market Category)
preprocesador = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline,     variables_numericas),
        ('cat', categorical_pipeline, variables_categoricas)
    ],
    remainder='drop'
)

In [10]:
# ==========================================
# PIPELINE COMPLETO
# ==========================================

pipeline_final = Pipeline([
    # Paso 1: Elimina 'Model' (915 valores únicos → 915 columnas con OHE → overfitting)
    ('drop_cols',      DropColumnsTransformer(columns_to_drop=['Model'])),

    # Paso 2: Convierte 'UNKNOWN' → NaN real (Transmission Type: 19 registros)
    ('clean_unknowns', UnknownToNaNTransformer()),

    # Paso 3: Elimina Market Category (31.4% nulos + valores multi-etiqueta)
    ('drop_high_nan',  DropHighMissingTransformer(threshold=0.25)),

    # Paso 4: Imputa nulos restantes con mediana/moda según tipo y porcentaje
    ('smart_imputer',  SmartImputerTransformer(low_threshold=0.10)),

    # Paso 5: Escala numéricas + codifica categóricas en columnas de 0 y 1
    ('preprocesamiento', preprocesador)
])

# Diagrama interactivo del pipeline (se ve en el notebook)
pipeline_final

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_cols', ...), ('clean_unknowns', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns_to_drop,['Model']
,threshold,0.25
,low_threshold,0.1
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_th

## 4. Ejecución del Pipeline

In [11]:
print('⚙️ Ejecutando el Pipeline...')
print(f'Dimensiones originales: {df_opt.shape}')

# fit_transform: aprende parámetros Y aplica transformación en un solo paso
# En producción real: fit() solo en X_train, transform() en X_train y X_test
matriz_procesada = pipeline_final.fit_transform(df_opt)

print(f'\n✅ ¡Éxito! Dimensiones finales: {matriz_procesada.shape}')
print('El aumento de columnas viene del OneHotEncoding de variables categóricas.')
print('\nVista previa de los primeros 3 registros (matriz lista para ML):')
print(matriz_procesada[:3])

⚙️ Ejecutando el Pipeline...
Dimensiones originales: (11914, 16)
🗑️  Columnas eliminadas (>25% nulos): ['Market Category']
🧠 SmartImputer - Simple (<10%): ['Engine Fuel Type', 'Engine HP', 'Engine Cylinders', 'Transmission Type', 'Number of Doors']
🚧 SmartImputer - Complejo (>10%): [] (PENDIENTE - futuro KNNImputer)

✅ ¡Éxito! Dimensiones finales: (11914, 92)
El aumento de columnas viene del OneHotEncoding de variables categóricas.

Vista previa de los primeros 3 registros (matriz lista para ML):
[[ 0.07039651  0.90128865  0.27374239 -1.63012179 -0.04025428 -0.01606936
   2.13295463  0.          0.          0.          0.          1.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          0.          0

In [12]:
# Recuperar nombres de columnas para que la matriz sea legible
enrutador = pipeline_final.named_steps['preprocesamiento']
nombres_finales = enrutador.get_feature_names_out()
nombres_limpios = [n.replace('num__','').replace('cat__','') for n in nombres_finales]

print(f'Total de features listas para el modelo: {len(nombres_limpios)}')

# Guardar el dataset procesado
df_final = pd.DataFrame(matriz_procesada, columns=nombres_limpios)
df_final.to_csv('data/processed/processed_data.csv', index=False)
print('✅ Dataset procesado guardado en data/processed/processed_data.csv')
print()
print('Vista del dataset procesado:')
df_final.head(3)

Total de features listas para el modelo: 92
✅ Dataset procesado guardado en data/processed/processed_data.csv

Vista del dataset procesado:


,Year,Engine HP,Engine Cylinders,Number of Doors,highway MPG,city mpg,Popularity,Make_Acura,Make_Alfa Romeo,Make_Aston Martin,...,Vehicle Style_Convertible,Vehicle Style_Convertible SUV,Vehicle Style_Coupe,Vehicle Style_Crew Cab Pickup,Vehicle Style_Extended Cab Pickup,Vehicle Style_Passenger Minivan,Vehicle Style_Passenger Van,Vehicle Style_Regular Cab Pickup,Vehicle Style_Sedan,Vehicle Style_Wagon
0,0.070397,0.901289,0.273742,-1.630122,-0.040254,-0.016069,2.132955,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.070397,0.547178,0.273742,-1.630122,0.278516,-0.016069,2.132955,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.070397,0.547178,0.273742,-1.630122,0.278516,0.188035,2.132955,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
print('=' * 55)
print('  RESUMEN DEL PIPELINE — CAR PRICE PROJECT')
print('=' * 55)
print(f'  Dataset original:    {df_raw.shape[0]:>6} filas × {df_raw.shape[1]} columnas')
print(f'  Dataset para ML:     {matriz_procesada.shape[0]:>6} filas × {matriz_procesada.shape[1]} features')
print(f'  Nulos en resultado:  {df_final.isnull().sum().sum()} (debe ser 0)')
print()
print('  Transformaciones aplicadas:')
print('  1. "Model" eliminada (915 valores → overfitting)')
print('  2. "UNKNOWN" → NaN en Transmission Type (19 registros)')
print('  3. "Market Category" eliminada (31.4% nulos + multi-etiqueta)')
print('  4. Outliers winsorisados con IQR (MSRP: $2M → $74K)')
print('  5. Nulos imputados: mediana (HP, Cylinders) / moda (Transmission, Doors)')
print('  6. Numéricas escaladas con StandardScaler')
print('  7. Categóricas codificadas con OneHotEncoder')
print('=' * 55)
print('  ✅ Datos listos para entrenar un modelo de regresión')
print('=' * 55)

  RESUMEN DEL PIPELINE — CAR PRICE PROJECT
  Dataset original:     11914 filas × 16 columnas
  Dataset para ML:      11914 filas × 92 features
  Nulos en resultado:  0 (debe ser 0)

  Transformaciones aplicadas:
  1. "Model" eliminada (915 valores → overfitting)
  2. "UNKNOWN" → NaN en Transmission Type (19 registros)
  3. "Market Category" eliminada (31.4% nulos + multi-etiqueta)
  4. Outliers winsorisados con IQR (MSRP: $2M → $74K)
  5. Nulos imputados: mediana (HP, Cylinders) / moda (Transmission, Doors)
  6. Numéricas escaladas con StandardScaler
  7. Categóricas codificadas con OneHotEncoder
  ✅ Datos listos para entrenar un modelo de regresión
